In [1]:
# Install dependecncies
%pip install anthropic python-dotenv

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 15.2 MB/s  0:00:007.1 MB/s eta 0:00:01
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [anthropic]━ 5/6 [anthropic]
Note: you may need to restart the kernel to use updated packages.


In [18]:
# Load env variables
from dotenv import load_dotenv
import json

load_dotenv()

True

In [11]:
# Create an API Client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [12]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, output_config=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if output_config:
        params["output_config"] = output_config
    
    
    response = client.messages.create(**params)

    text = next(block.text for block in response.content if block.type == "text")
    return text

In [19]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)

    text = chat(messages, output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "array",
                "properties": {
                    "task": {"type": "string"},
                },
                "required": ["task"],
                "additionalProperties": False,
            },
        }
    })

    return json.loads(text)

In [13]:
dataset = generate_dataset()

dataset

[{'task': 'Write a Python function using boto3 that takes an S3 bucket name and returns the total number of objects in the bucket, handling pagination.'},
 {'task': "Write a JSON IAM policy document that allows read-only access (s3:GetObject and s3:ListBucket) to the bucket named 'analytics-logs' and denies all delete actions."},
 {'task': "Write a regex that matches valid AWS EC2 instance IDs, which begin with 'i-' followed by either 8 or 17 hexadecimal characters."}]

In [14]:
with open("dataset.json", "w") as file:
    json.dump(dataset, file, indent=2)

In [13]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}

    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)

    return output

In [27]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages, output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "strengths": {
                        "type": "array",
                        "items": {"type": "string"},
                        "minItems": 1,
                    },
                    "weaknesses": {
                        "type": "array",
                        "items": {"type": "string"},
                        "minItems": 1,
                    },
                    "reasoning": {
                        "type": "string"
                    },
                    "score": {
                        "type": "number",
                    }
                },
                "required": [
                    "strengths",
                    "weaknesses",
                    "reasoning",
                    "score"
                ],
                "additionalProperties": False
            }
        }
    })
    return json.loads(eval_text)

In [28]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:
from statistics import mean

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

In [30]:
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [31]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a Python function that uses boto3 to count all objects in an S3 bucket, using a paginator so it correctly handles buckets with more than 1,000 objects (the default page size limit for `list_objects_v2`):\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\n\n\ndef count_s3_objects(bucket_name: str) -> int:\n    \"\"\"\n    Return the total number of objects in an S3 bucket.\n\n    Uses a paginator to iterate through all pages of results, so it works\n    correctly for buckets containing more than 1,000 objects.\n\n    Args:\n        bucket_name: The name of the S3 bucket.\n\n    Returns:\n        The total number of objects in the bucket.\n\n    Raises:\n        ClientError: If the bucket doesn't exist or access is denied.\n    \"\"\"\n    s3 = boto3.client(\"s3\")\n    paginator = s3.get_paginator(\"list_objects_v2\")\n\n    total = 0\n    try:\n        for page in paginator.paginate(Bucket=bucket_name):\n            # \"Contents\" is ab